# AI Energy Theft Detection System

**Context**: Distribution theft and meter bypassing cause massive non-technical losses for utility companies. Discovering these anomalies traditionally requires expensive manual audits and "truck rolls."

In this demonstration, we simulate a scenario where a utility company provides us with their customer database. This database contains:
1. The **Geospatial Coordinates** of the customer's property.
2. The **Reported Monthly Energy Consumption (kWh)**.
3. Regional **Blackout Frequency Data**.

Our Multimodal AI pipeline will use these coordinates to autonomously fetch satellite imagery, determine the true physical footprint of the building, and identify mathematically suspicious discrepancies between the building's physical size and its reported consumption.


## 1. Setup & Environment
First, we install the necessary libraries for cloud API access, image processing, and local AI modeling.


In [ ]:
!pip install google-genai requests pillow pandas transformers torch kagglehub accelerate --quiet


In [ ]:
import os
import pandas as pd
import numpy as np
import random
import requests
import json
import torch
import kagglehub
from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image
from google import genai

# Try to load credentials from Kaggle Secrets if available
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
    os.environ["MAPS_API_KEY"] = UserSecretsClient().get_secret("MAPS_API_KEY")
except Exception:
    print("Please ensure GEMINI_API_KEY and MAPS_API_KEY are set.")

DATA_DIR = 'data'
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)



## 2. Simulating the Utility Database
In a real-world scenario, this data is exported directly from the utility's AMI (Advanced Metering Infrastructure) or billing software. 
For this notebook, we will simulate this database. We will use the Overpass API to fetch real building coordinates in Lagos, Nigeria, and then generate synthetic usage data, purposefully simulating "theft" for a small percentage of them.


In [ ]:
NUM_BUILDINGS = 50 # Kept small for demonstration purposes
LAGOS_LAT = 6.5244
LAGOS_LNG = 3.3792

def fetch_real_building_coordinates(limit=50):
    print("Fetching real building coordinates from Overpass API (Simulating Utility Database)...")
    overpass_url = "http://overpass-api.de/api/interpreter"
    fetch_amount = limit * 3 
    
    overpass_query = f"""
    [out:json][timeout:25];
    way["building"](6.4, 3.3, 6.6, 3.5);
    out {fetch_amount} center;
    """
    try:
        headers = {'User-Agent': 'EnergyTheftSystem/1.0'}
        response = requests.post(overpass_url, data={'data': overpass_query}, headers=headers)
        response.raise_for_status()
        data = response.json()
        coords = []
        for element in data.get('elements', []):
            if 'center' in element:
                coords.append((element['center']['lat'], element['center']['lon']))
        
        if len(coords) < fetch_amount:
            while len(coords) < fetch_amount:
                coords.append((LAGOS_LAT + random.uniform(-0.05, 0.05), LAGOS_LNG + random.uniform(-0.05, 0.05)))
                
        random.shuffle(coords)
        return coords
    except Exception as e:
        print(f"Failed to fetch from Overpass API: {e}. Falling back to random coords.")
        return [(LAGOS_LAT + random.uniform(-0.05, 0.05), LAGOS_LNG + random.uniform(-0.05, 0.05)) for _ in range(fetch_amount)]

# Generate the mock utility data
building_coords = fetch_real_building_coordinates(NUM_BUILDINGS)
print(f"Acquired {len(building_coords)} coordinate pairs.")



## 3. Remote Visual Verification (The Pipeline)
Now the actual analysis begins. We iterate through the utility-provided coordinates.
For each coordinate, we:
1. Download a top-down satellite image.
2. Use a local **Gemma-4 Vision Model** to verify the image is valid.
3. Use the same model to classify the physical footprint (Size, Property Tag, Neighborhood).


In [ ]:
def fetch_satellite_image(lat, lng, building_id):
    image_path = os.path.join(IMAGES_DIR, f"{building_id}.jpg")
    if os.path.exists(image_path): return image_path
        
    maps_api_key = os.getenv("MAPS_API_KEY")
    if not maps_api_key: return None
        
    url = f"https://maps.googleapis.com/maps/api/staticmap?center={lat},{lng}&zoom=21&size=400x400&maptype=satellite&key={maps_api_key}"
    response = requests.get(url)
    if response.status_code == 200:
        with open(image_path, 'wb') as f:
            f.write(response.content)
        return image_path
    return None

def validate_image_with_gemini(processor, model, image_path):
    if not processor or not model or not image_path: return True
    try:
        img = Image.open(image_path).convert("RGB")
        prompt = "Is this a clear, top-down satellite view of a single primary building? Answer only YES or NO."
        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=text, images=img, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]
        outputs = model.generate(**inputs, max_new_tokens=10)
        response_text = processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip().upper()
        return "YES" in response_text
    except: return True

def analyze_image_with_gemini(processor, model, image_path):
    if not processor or not model or not image_path:
        return {"building_size_category": random.choice(['Small', 'Medium', 'Large', 'Huge']), "property_tag": random.choice(['Residential', 'Commercial', 'Industrial']), "neighborhood_type": random.choice(['Slum', 'Dense Urban', 'Suburb', 'Commercial District'])}
    try:
        img = Image.open(image_path).convert("RGB")
        prompt = """Analyze this satellite image of a building in Lagos, Nigeria.
Return ONLY a JSON object with the following three keys exactly:
- "building_size_category": one of ["Small", "Medium", "Large", "Huge"]
- "property_tag": one of ["Residential", "Commercial", "Industrial"]
- "neighborhood_type": one of ["Slum", "Dense Urban", "Suburb", "Commercial District"]"""
        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=text, images=img, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]
        outputs = model.generate(**inputs, max_new_tokens=150)
        response_text = processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
        if response_text.startswith("```json"): response_text = response_text[7:]
        if response_text.endswith("```"): response_text = response_text[:-3]
        return json.loads(response_text)
    except:
        return {"building_size_category": "Medium", "property_tag": "Residential", "neighborhood_type": "Dense Urban"}



We also need a helper to generate the synthetic utility data (Consumption and Blackouts) since we don't have a real utility database. We will simulate theft for 5% of our targets.


In [ ]:
def generate_energy_and_blackout_data(vision_data):
    size = vision_data.get('building_size_category', 'Medium')
    tag = vision_data.get('property_tag', 'Residential')
    neighborhood = vision_data.get('neighborhood_type', 'Dense Urban')
    
    size_multiplier = {'Small': 50, 'Medium': 150, 'Large': 400, 'Huge': 1000}
    tag_multiplier = {'Residential': 1.0, 'Commercial': 2.5, 'Industrial': 5.0}
    base_usage = size_multiplier.get(size, 150) * tag_multiplier.get(tag, 1.0)
    monthly_usage_kwh = round(base_usage * random.uniform(0.8, 1.2), 2)
    
    if tag == 'Residential': load_type = random.choices(['Morning/Evening Peak', 'Constant', 'Erratic'], weights=[0.7, 0.2, 0.1])[0]
    elif tag == 'Commercial': load_type = random.choices(['Daytime Peak', 'Constant'], weights=[0.8, 0.2])[0]
    else: load_type = 'Constant Heavy'
        
    if neighborhood == 'Slum': blackout_freq, blackout_dur = random.randint(15, 30), random.randint(4, 12)
    elif neighborhood == 'Dense Urban': blackout_freq, blackout_dur = random.randint(5, 15), random.randint(2, 6)
    elif neighborhood == 'Commercial District': blackout_freq, blackout_dur = random.randint(1, 5), random.randint(1, 3)
    else: blackout_freq, blackout_dur = random.randint(2, 10), random.randint(2, 5)
        
    return {"monthly_usage_kwh": monthly_usage_kwh, "load_type": load_type, "blackout_frequency": blackout_freq, "average_blackout_duration_hrs": blackout_dur}



### Loading the Local Vision Model
We load the `Gemma-4-E2B` model into Kaggle's GPU memory.


In [ ]:
print("Loading Local Gemma-4-E2B Model...")
try:
    MODEL_PATH = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e2b")
    processor = AutoProcessor.from_pretrained(MODEL_PATH)
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.bfloat16, device_map="auto")
    print("Model loaded successfully!")
except Exception as e:
    print(f"Failed to load model locally: {e}")
    processor, model = None, None



### Executing the Data Pipeline


In [ ]:
buildings = []
num_theft = int(NUM_BUILDINGS * 0.05)
theft_indices = set(random.sample(range(NUM_BUILDINGS), num_theft))

print(f"Processing {NUM_BUILDINGS} utility locations...")
valid_buildings_count = 0
coord_index = 0

while valid_buildings_count < NUM_BUILDINGS and coord_index < len(building_coords):
    building_id = f'B{valid_buildings_count:04d}'
    lat, lng = building_coords[coord_index]
    coord_index += 1
    
    image_path = fetch_satellite_image(lat, lng, building_id)
    if not validate_image_with_gemini(processor, model, image_path):
        if os.path.exists(image_path): os.remove(image_path)
        continue
        
    vision_data = analyze_image_with_gemini(processor, model, image_path)
    energy_data = generate_energy_and_blackout_data(vision_data)
    
    is_theft = valid_buildings_count in theft_indices
    monthly_usage = energy_data['monthly_usage_kwh']
    if is_theft:
        monthly_usage = round(monthly_usage * random.uniform(0.1, 0.2), 2) # Simulate theft
    
    building_record = {
        'building_id': building_id,
        'lat': lat, 'lng': lng,
        'building_size_category': vision_data.get('building_size_category'),
        'property_tag': vision_data.get('property_tag'),
        'neighborhood_type': vision_data.get('neighborhood_type'),
        'monthly_usage_kwh': monthly_usage,
        'load_type': energy_data['load_type'],
        'blackout_frequency': energy_data['blackout_frequency'],
        'average_blackout_duration_hrs': energy_data['average_blackout_duration_hrs'],
    }
    buildings.append(building_record)
    valid_buildings_count += 1
    if valid_buildings_count % 10 == 0: print(f"Processed {valid_buildings_count}/{NUM_BUILDINGS}")

df = pd.DataFrame(buildings)



## 4. Statistical Peer Analysis
With the physical traits verified by AI, we group buildings into peer sets (e.g., "Large Industrial").
We calculate the mean consumption for each peer group and assign a Z-score to every building. If a building's usage is drastically below its peers, it is flagged with a high Risk Score.


In [ ]:
print("Calculating statistical baselines...")
group_stats = df.groupby(['building_size_category', 'property_tag'])['monthly_usage_kwh'].agg(['mean', 'std']).reset_index()
df = df.merge(group_stats, on=['building_size_category', 'property_tag'], how='left')
df['std'] = df['std'].fillna(1.0).replace(0, 1.0)
df['z_score'] = (df['monthly_usage_kwh'] - df['mean']) / df['std']

def calculate_risk(row):
    score = 10.0
    reasons = []
    if row['z_score'] < -2.0:
        score += 80.0
        reasons.append(f"Suspiciously low usage. More than 2 std devs below average ({round(row['mean'], 1)} kWh) for {row['building_size_category']} {row['property_tag']} buildings.")
    elif row['z_score'] < -1.0:
        score += 40.0
        reasons.append(f"Noticeably below average ({round(row['mean'], 1)} kWh) for {row['building_size_category']} {row['property_tag']} buildings.")
    if row['load_type'] == 'Erratic' and row['property_tag'] != 'Industrial':
        score += 20.0
        reasons.append("Erratic load type is unusual for this property tag.")
    if row['blackout_frequency'] > 15:
        score += 10.0
        
    score = min(100.0, score + random.uniform(-5, 5))
    reason = " ".join(reasons) if reasons else "Usage is within normal expected range compared to peers."
    return pd.Series([round(score, 1), reason])

df[['risk_score', 'anomaly_reason']] = df.apply(calculate_risk, axis=1)
df = df.rename(columns={'mean': 'expected_monthly_usage_kwh'})
df = df.drop(columns=['std', 'z_score'])
df.to_csv(os.path.join(DATA_DIR, 'buildings.csv'), index=False)
print("Analysis complete. Outliers flagged.")



## 5. Automated Forensic Synthesis
Finally, we take the highest-risk building flagged by our statistical analysis and feed its complete profile—along with its satellite image—into a high-capacity reasoning model (**Gemini**).
This step generates a detailed, evidence-backed report explaining *why* the building is suspicious, giving utility inspectors actionable intelligence before they dispatch a truck.


In [ ]:
high_risk_df = df[df['risk_score'] > 50].sort_values(by='risk_score', ascending=False)

if not high_risk_df.empty:
    target = high_risk_df.iloc[0].to_dict()
    print(f"=== FLAG IDENTIFIED ===")
    print(f"Target ID: {target['building_id']}")
    print(f"Risk Score: {target['risk_score']}%")
    print(f"Reported Usage: {target['monthly_usage_kwh']} kWh\n")
    
    image_path = f"data/images/{target['building_id']}.jpg"
    image = None
    if os.path.exists(image_path): image = Image.open(image_path).convert("RGB")

    prompt = f"""You are an AI Energy Theft Detection forensics analyst.
I have attached a satellite image of a building.
Step 1: Look at the image to verify its physical attributes. You must classify its size as one of [Small, Medium, Large, Huge] and its tag as one of [Residential, Commercial, Industrial].
Step 2: Acknowledge the utility baseline data reported below.
Step 3: Compare the reported consumption data with average expected data for addresses with a similar size and tag in this region.
Step 4: Factor in the blackout frequency for that region. NOTE: High blackout frequency is often a strong indicator of localized energy theft, as unmetered stolen consumption overloads neighborhood transformers.
Step 5: Synthesize a refined, numbers-backed analysis on the probability of distribution theft (bypassing the meter).

Utility Baseline Data:
- ID: {target['building_id']}
- Reported Size Category: {target['building_size_category']}
- Reported Property Tag: {target['property_tag']}
- Neighborhood Type: {target['neighborhood_type']}
- Reported Monthly Usage: {target['monthly_usage_kwh']} kWh
- Load Type: {target['load_type']}
- Regional Blackout Frequency: {target['blackout_frequency']} times/month
- Regional Avg Blackout Duration: {target['average_blackout_duration_hrs']} hrs
- Pre-calculated Risk Score: {target['risk_score']}%
- Base Heuristic Reason: {target['anomaly_reason']}

Provide a single paragraph explanation using this 5-step logic. You MUST explicitly quote the provided numeric data points in your reasoning so the conclusion is strictly data-backed and not just a generic summary.
"""
    
    try:
        client = genai.Client()
        contents = [image, prompt] if image else [prompt]
        response = client.models.generate_content(
            model="gemma-4-26b-a4b-it",
            contents=contents,
        )
        print("=== AUTOMATED FORENSIC REPORT ===")
        print(response.text.strip())
    except Exception as e:
        print(f"Could not reach Gemini API for reporting. Error: {e}")
else:
    print("No high risk buildings found.")



## 6. Financial Impact & High-Volume Prioritization
Risk scores tell us the probability of theft, but not the scale. Here we calculate the estimated absolute stolen volume (MWh) by comparing the reported usage with the expected peer baseline for high-risk buildings. This allows the utility to prioritize investigations by maximum revenue impact.

In [ ]:
impact_df = high_risk_df.copy()
if not impact_df.empty and 'expected_monthly_usage_kwh' in impact_df.columns:
    impact_df['estimated_stolen_mwh'] = (impact_df['expected_monthly_usage_kwh'] - impact_df['monthly_usage_kwh']) / 1000.0
    impact_df = impact_df[impact_df['estimated_stolen_mwh'] > 0]
    impact_df = impact_df.sort_values(by='estimated_stolen_mwh', ascending=False)
    
    print("=== TOP 5 FINANCIAL IMPACT TARGETS ===\n")
    for idx, row in impact_df.head(5).iterrows():
        print(f"Target ID: {row['building_id']}")
        print(f"Risk Score: {row['risk_score']}%")
        print(f"Expected Usage: {round(row['expected_monthly_usage_kwh'], 1)} kWh")
        print(f"Reported Usage: {row['monthly_usage_kwh']} kWh")
        print(f"-> Estimated Stolen Volume: {round(row['estimated_stolen_mwh'], 3)} MWh/month\n")
else:
    print("No data available for impact analysis.")
